## Concise, barebones workflow for getting to a TSM Line

#### Imports

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import pandas as pd
import numpy as np
import scipy as sci
import matplotlib.pyplot as pl
import matplotlib.image as img
import subprocess
import pathlib
import pyvista as pv
import copy

import fenics_sz.utils
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry
from fenics_sz.sz_problems.sz_steady_dislcreep import SteadyDislSubductionProblem
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem
from fenics_sz.sz_problems.sz_params import default_params, allsz_params

In [ ]:
from fenics_sz.fluid_release.perple_x_integration import get_PT_data_from_tabs, plot_PT_data
import fenics_sz.fluid_release.get_PT_curves 

In [ ]:
from fenics_sz.fluid_release.workflow_functions import (in_domain, get_st_grid, get_interpolator, predict_h2o,
                                                          Cell, remove_rehydration, get_water_loss, sorted_water_loss_by_layer,
                                                          get_TSMstye_line)

#### Read Perple_X data

In [ ]:
DMM_data = get_PT_data_from_tabs('DMMdamp_25')
uvolcs_data = get_PT_data_from_tabs('upvolc_25')
lvolcs_data = get_PT_data_from_tabs('lovolc_25')
dikes_data = get_PT_data_from_tabs('dike_25')
gabbros_data = get_PT_data_from_tabs('gabbro_25')

In [ ]:
interps = []
interps.append(get_interpolator(uvolcs_data))
interps.append(get_interpolator(lvolcs_data))
interps.append(get_interpolator(dikes_data))
interps.append(get_interpolator(gabbros_data))
interps.append(get_interpolator(DMM_data))

### Run Workflow on Abers Examples

In [ ]:
resscale = 5.0
u_res = 100
h_serp = 2.0

#### Nicaragua

In [ ]:
dirname = "07_Nicaragua"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "Carbonate41_25"

In [ ]:
get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

#### Alaska Peninsula

In [ ]:
dirname = "01_Alaska_Peninsula"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "AlaskaTurb46_25"

In [ ]:
get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

#### North Honshu

In [ ]:
dirname = "48_N_Honshu"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "Diatom44_25"

In [ ]:
get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

#### Cascadia

In [ ]:
dirname = "04_Cascadia"
sz_dict = allsz_params[dirname]
sz_dict['sediment'] = "Pelagic45_25"

get_TSMstye_line(sz_dict, h_serp, u_res, resscale, interps)

Vast differences in sediments in alaska peninsula and cascadia cases. I checked the input files for the sediments against what Geoff used since that's where the discrepancies lie, and they seemed fine

To do:
* get function to output data for plotting a tsm lite
* workflow for (G)WR calculations
* look into sediments, edit the json files